# 03 - Grad-CAM Interpretability Review

This notebook reviews Grad-CAM overlays generated by `src/interpret.py` for both model families.

Focus:
- verify artifacts exist,
- inspect qualitative attention behavior,
- record concise interpretation notes for the final report.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
from IPython.display import display, Markdown, Image

REPO_NAME = "CS-171-Chest-X-Ray-Medical-Diagnosis"
REPO_URL = "https://github.com/jenilkathrotia/CS-171-Chest-X-Ray-Medical-Diagnosis.git"
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    colab_repo = Path("/content") / REPO_NAME
    if not colab_repo.exists():
        subprocess.run(["git", "clone", REPO_URL, str(colab_repo)], check=True)
    os.chdir(colab_repo)


def resolve_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate

    colab_candidate = Path("/content") / REPO_NAME
    if (colab_candidate / "src").exists():
        return colab_candidate

    raise FileNotFoundError(
        f"Could not locate project root from cwd={start}. Expected a directory containing src/ and notebooks/."
    )


cwd = Path.cwd()
project_root = resolve_project_root(cwd)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

if IS_COLAB and (project_root / "requirements.txt").exists():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(project_root / "requirements.txt")],
        check=False,
    )

GRADCAM_DIR = project_root / "results" / "gradcam"
custom_imgs = sorted(GRADCAM_DIR.glob("custom_cnn_gradcam_*.png"))
dense_imgs = sorted(GRADCAM_DIR.glob("densenet121_gradcam_*.png"))

print("cwd:", cwd)
print("project_root:", project_root)
print("Grad-CAM directory:", GRADCAM_DIR)
print("custom_cnn images:", len(custom_imgs))
print("densenet121 images:", len(dense_imgs))

if not custom_imgs and not dense_imgs:
    print(
        "No Grad-CAM images found. Generate them with `python -m src.interpret gradcam --model custom_cnn --data-dir <path>` and `--model densenet121` first."
    )

In [ ]:
display(Markdown("## Custom CNN Grad-CAM Samples"))
for img_path in custom_imgs[:6]:
    display(Markdown(f"### {img_path.name}"))
    display(Image(filename=str(img_path)))

display(Markdown("## DenseNet121 Grad-CAM Samples"))
for img_path in dense_imgs[:6]:
    display(Markdown(f"### {img_path.name}"))
    display(Image(filename=str(img_path)))

In [ ]:
display(Markdown("## Interpretability Commentary"))
display(
    Markdown(
        "\n".join([
            "- In the current run, both models produce highly similar prediction behavior; Grad-CAM overlays should be interpreted alongside metric outputs.",
            "- If attention appears diffuse or non-localized, this indicates weak class-discriminative features in the current checkpoints.",
            "- For final clinical interpretation, pair Grad-CAM findings with per-class recall/F1 and confusion matrix outcomes rather than using heatmaps alone.",
            "- Recommended follow-up for a stronger report: include representative correct/incorrect cases from both classes and discuss failure patterns."
        ])
    )
)